## Load the required packages

In [1]:
import numpy as np
import torch.nn as nn
import pandas as pd
import torch.nn.functional as F
import torch
import torch.optim as optim
from scipy import stats
import anndata
import os
from scipy.sparse import csr_matrix
from pathlib import Path

## Set path

In [2]:
working_dir = Path(os.getcwd())
dataset_dir = f"{working_dir.parents[1]}/datasets/CITE-SLN-Gayoso"
base_dir = f"{working_dir.parents[1]}/outputs/batch variation/208_M2-208_M1/cTPnet"
os.makedirs(base_dir, exist_ok=True)

## Load data

In [3]:
train_data = anndata.read(f"{dataset_dir}/SLN208_M2.h5ad")  
test_data = anndata.read(f"{dataset_dir}/SLN208_M1.h5ad")   
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

In [4]:
denoised_gene_expression_train = pd.read_table(f"{dataset_dir}/SLN208_M2_denoised_gene_expression_data.txt", index_col=0, sep=" ").T
denoised_gene_expression_test = pd.read_table(f"{dataset_dir}/SLN208_M1_denoised_gene_expression_data.txt", index_col=0, sep=" ").T
denoised_gene_expression_train, denoised_gene_expression_test

(       1      2      3      4      5      6      7      8      9      10     \
 V1     0.138  0.130  0.218  0.063  0.179  0.033  0.144  0.010  0.037  0.001   
 V2     0.083  0.076  0.309  0.071  0.252  0.043  0.214  0.011  0.040  0.001   
 V3     0.118  0.144  0.275  0.080  0.312  0.042  0.231  0.011  0.025  0.001   
 V4     0.104  0.116  0.516  0.061  0.343  0.041  0.388  0.011  0.037  0.001   
 V5     0.081  0.089  0.293  0.058  0.218  0.039  0.250  0.011  0.030  0.001   
 ...      ...    ...    ...    ...    ...    ...    ...    ...    ...    ...   
 V7101  0.133  0.126  0.390  0.058  0.260  0.045  0.512  0.011  0.037  0.001   
 V7102  0.113  0.103  0.228  0.079  0.293  0.047  0.243  0.010  0.028  0.001   
 V7103  0.120  0.146  0.251  0.100  0.405  0.068  0.294  0.011  0.029  0.001   
 V7104  0.097  0.100  0.313  0.061  0.317  0.039  0.311  0.010  0.019  0.001   
 V7105  0.123  0.200  0.242  0.075  0.330  0.052  0.248  0.010  0.026  0.001   
 
        ...  13544  13545  13546  1354

In [5]:
denoised_gene_expression_train.index = train_data.obs.index
denoised_gene_expression_train.columns = train_data.var.index
denoised_gene_expression_test.index = test_data.obs.index
denoised_gene_expression_test.columns = test_data.var.index
denoised_gene_expression_train, denoised_gene_expression_test

(index               Mrpl15  Lypla1  Tcea1  Atp6v1h  Rb1cc1  4732440D04Rik  \
 index                                                                       
 AAACCCAAGGTATCTC-2   0.138   0.130  0.218    0.063   0.179          0.033   
 AAACCCACATTCTGTT-2   0.083   0.076  0.309    0.071   0.252          0.043   
 AAACCCAGTAGCTTGT-2   0.118   0.144  0.275    0.080   0.312          0.042   
 AAACCCAGTATCAGCT-2   0.104   0.116  0.516    0.061   0.343          0.041   
 AAACCCAGTTTCTATC-2   0.081   0.089  0.293    0.058   0.218          0.039   
 ...                    ...     ...    ...      ...     ...            ...   
 TTTGTTGGTGGGCTCT-2   0.133   0.126  0.390    0.058   0.260          0.045   
 TTTGTTGTCAAATAGG-2   0.113   0.103  0.228    0.079   0.293          0.047   
 TTTGTTGTCACGAGGA-2   0.120   0.146  0.251    0.100   0.405          0.068   
 TTTGTTGTCTCGACCT-2   0.097   0.100  0.313    0.061   0.317          0.039   
 TTTGTTGTCTTGGTCC-2   0.123   0.200  0.242    0.075   0.330     

## Select common genes

In [6]:
common_genes = np.intersect1d(train_data.var.index, test_data.var.index)
train_data.X = denoised_gene_expression_train.loc[:, common_genes]
test_data.X = denoised_gene_expression_test.loc[:, common_genes]

## Select common proteins

In [7]:
common_proteins = np.intersect1d(np.array(train_data.uns["protein_names"]), np.array(test_data.uns["protein_names"]))
if isinstance(train_data.obsm["protein_expression"], pd.DataFrame):
    train_protein_expression = train_data.obsm["protein_expression"]
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = train_protein_expression
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)
else:
    train_protein_expression = pd.DataFrame(train_data.obsm["protein_expression"].todense(), index=train_data.obs.index, columns=train_data.uns["protein_names"])
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = csr_matrix(train_protein_expression.values)
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)

if isinstance(test_data.obsm["protein_expression"], pd.DataFrame):
    test_protein_expression = test_data.obsm["protein_expression"]
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = test_protein_expression
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)
else:
    test_protein_expression = pd.DataFrame(test_data.obsm["protein_expression"].todense(), index=test_data.obs.index, columns=test_data.uns["protein_names"])
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = csr_matrix(test_protein_expression.values)
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)

In [8]:
gene_list = list(set(train_data.var.index.tolist()))
gene_list.sort()
X = train_data.to_df().loc[:, gene_list]   
x_batch2 = test_data.to_df().loc[:, gene_list]

In [9]:
protein_list = list(set(train_data.uns["protein_names"]))
y = train_protein_expression.loc[X.index, protein_list]

## Replace '.' with '/' for PyTorch and record the mapping

In [10]:
if any("." in item for item in protein_list):
    protein_names_convert = pd.DataFrame({"original_name": protein_list})
    protein_list = [item.replace('.', '/') for item in protein_list]
    protein_names_convert["convert_name"] = protein_list
    protein_names_convert.to_csv(f"{base_dir}/cTPnet_protein_names_convert.csv")
protein_names_convert

,original_name,convert_name
0,ADT_CD339(Jagged1)_A0887,ADT_CD339(Jagged1)_A0887
1,ADT_CD272(BTLA)_A0881,ADT_CD272(BTLA)_A0881
2,ADT_ESAM_A0596,ADT_ESAM_A0596
3,ADT_CD63_A0559,ADT_CD63_A0559
4,ADT_CD200(OX2)_A0079,ADT_CD200(OX2)_A0079
...,...,...
193,ADT_CD83_A0562,ADT_CD83_A0562
194,ADT_CD26(DPP-4)_A0883,ADT_CD26(DPP-4)_A0883
195,ADT_ENPP1_A0891,ADT_ENPP1_A0891
196,ADT_CD150(SLAM)_A0203,ADT_CD150(SLAM)_A0203


## Normalize reference raw gene and protein expression

In [11]:
X = X.apply(lambda x: np.log((x * 10000.0 / sum(x)) + 1.0), axis=1)   
y = y.apply(lambda x: np.log((x + 1.0) / stats.gmean(x + 1.0)), axis=1)   
X, y

(index               0610007P14Rik  0610009B22Rik  0610009L18Rik  \
 index                                                             
 AAACCCAAGGTATCTC-2       0.421835       0.471478       2.961442   
 AAACCCACATTCTGTT-2       0.437466       0.537453       2.490818   
 AAACCCAGTAGCTTGT-2       0.436213       0.336299       2.785374   
 AAACCCAGTATCAGCT-2       0.510199       0.343316       2.447741   
 AAACCCAGTTTCTATC-2       0.501419       0.529970       2.548012   
 ...                           ...            ...            ...   
 TTTGTTGGTGGGCTCT-2       0.477034       0.314193       2.493733   
 TTTGTTGTCAAATAGG-2       0.358082       0.347274       2.686135   
 TTTGTTGTCACGAGGA-2       0.325289       0.427981       2.342618   
 TTTGTTGTCTCGACCT-2       0.420162       0.320465       2.573154   
 TTTGTTGTCTTGGTCC-2       0.373719       0.395944       2.651654   
 
 index               0610009O20Rik  0610010F05Rik  0610010K14Rik  \
 index                                        

## Randomly select 15% of reference cells as validation set

In [12]:
X = X.sample(frac=1, random_state=4905)   
X_test = X.sample(frac=0.15, random_state=4905)   
y_test = y.loc[X_test.index, :] 
X_test, y_test

(index               0610007P14Rik  0610009B22Rik  0610009L18Rik  \
 index                                                             
 ACAAGCTCAGTTGAAA-2       0.446054       0.456211       2.262125   
 ACCAACAGTAATGATG-2       0.156092       0.358875       2.764660   
 GGGTAGATCACTGATG-2       0.539065       0.476321       2.305048   
 TTGTTCACATGCGGTC-2       0.217863       0.325080       2.883649   
 ATCCACCTCGTGTCAA-2       0.404903       0.293960       1.553570   
 ...                           ...            ...            ...   
 AGGCATTAGTTGGCGA-2       0.569470       0.636631       2.583740   
 TGTAAGCAGGGATGTC-2       0.335842       0.316824       3.052788   
 TCGATTTAGTCATCGT-2       0.442346       0.536774       2.623572   
 TTGGGATCATCTAGAC-2       0.638956       0.962203       2.301170   
 CAGAGCCGTAGGGAGG-2       0.417374       0.673065       2.611140   
 
 index               0610009O20Rik  0610010F05Rik  0610010K14Rik  \
 index                                        

In [13]:
X = X.drop(X_test.index)   
y = y.drop(y_test.index)   
y = y.loc[X.index, :]   
X, y

(index               0610007P14Rik  0610009B22Rik  0610009L18Rik  \
 index                                                             
 CAGGGCTAGAGTTGCG-2       0.260179       0.339209       3.141996   
 CTGTCGTCAACAGATA-2       0.036023       0.183768       2.749047   
 AGGTAGGTCGTTACCC-2       0.266208       0.297080       2.932774   
 AACCTTTAGAGCACTG-2       0.237665       0.375749       2.876952   
 GTATTGGAGTGCGTCC-2       0.300299       0.626066       2.366092   
 ...                           ...            ...            ...   
 AGAACAAGTTACCCAA-2       0.255741       0.270103       2.979447   
 AGTGATCTCAAGGTGG-2       0.438720       0.528180       2.988067   
 TGGAGGACAGTGTGGA-2       0.398481       0.403486       2.485387   
 GACGCTGCATCGAAGG-2       0.373597       0.383767       2.959528   
 ATGCATGCAACCACAT-2       0.389902       0.604653       2.662775   
 
 index               0610009O20Rik  0610010F05Rik  0610010K14Rik  \
 index                                        

## Convert gene and protein expression to Torch Tensors

In [14]:
X = torch.tensor(X.values)
X = X.type(torch.FloatTensor)
X = X.to("cuda")
y = torch.tensor(y.values)
y = y.type(torch.FloatTensor)
y = y.to("cuda")
X, y

(tensor([[0.2602, 0.3392, 3.1420,  ..., 0.3765, 0.4783, 4.2419],
         [0.0360, 0.1838, 2.7490,  ..., 0.1868, 0.1369, 4.1816],
         [0.2662, 0.2971, 2.9328,  ..., 0.3403, 0.1484, 4.6158],
         ...,
         [0.3985, 0.4035, 2.4854,  ..., 0.3833, 0.1522, 4.7369],
         [0.3736, 0.3838, 2.9595,  ..., 0.4569, 0.1449, 4.5749],
         [0.3899, 0.6047, 2.6628,  ..., 0.4181, 0.1563, 4.6640]],
        device='cuda:0'),
 tensor([[ 0.9858, -0.5547, -0.2670,  ...,  0.9116,  0.5439, -0.2670],
         [ 0.5642, -0.2243, -0.2243,  ..., -0.2243, -0.4474, -0.2243],
         [ 0.6826,  0.5773, -0.9268,  ...,  0.4595,  0.4595,  0.4595],
         ...,
         [ 1.0219,  1.0908, -0.9241,  ...,  0.7807,  1.0908, -0.5186],
         [ 1.1255, -0.0982, -0.6091,  ...,  0.4896, -0.6091, -0.3214],
         [ 0.4198,  0.8253,  0.1321,  ...,  0.1321,  1.5593, -0.9665]],
        device='cuda:0'))

In [15]:
X_test = torch.tensor(X_test.values).to("cuda")
X_test = X_test.type(torch.FloatTensor)
X_test = X_test.to("cuda")
y_test = torch.tensor(y_test.values).to("cuda")
y_test = y_test.type(torch.FloatTensor)
y_test = y_test.to("cuda")
X_test, y_test

(tensor([[0.4461, 0.4562, 2.2621,  ..., 0.4787, 0.1616, 4.7438],
         [0.1561, 0.3589, 2.7647,  ..., 0.3667, 0.1496, 4.4955],
         [0.5391, 0.4763, 2.3050,  ..., 0.4811, 0.1569, 4.2747],
         ...,
         [0.4423, 0.5368, 2.6236,  ..., 0.3218, 0.1531, 4.5405],
         [0.6390, 0.9622, 2.3012,  ..., 0.5675, 0.1618, 4.6492],
         [0.4174, 0.6731, 2.6111,  ..., 0.4433, 0.1615, 4.5318]],
        device='cuda:0'),
 tensor([[ 0.9256,  0.4401,  0.1524,  ...,  0.7586, -0.0299, -1.6393],
         [ 0.7549, -0.1205, -0.1205,  ...,  0.0618,  0.4672,  0.6679],
         [ 0.6685,  0.3119, -0.5354,  ..., -0.2478,  1.3617, -0.9409],
         ...,
         [ 1.0774,  1.0774, -0.1013,  ...,  0.8150,  1.4569, -1.4876],
         [ 0.6855,  0.1549, -0.3560,  ...,  0.6249,  0.4913,  0.1549],
         [ 0.7129, -0.3857, -0.6734,  ...,  0.4253,  2.5185, -0.3857]],
        device='cuda:0'))

## Define model network

In [16]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # self.fc1 = nn.Linear(14505, 1000)
        self.fc1 = nn.Linear(X.shape[1], 1000)
        self.fc2 = nn.Linear(1000, 256)
        self.fc3 = nn.ModuleDict({})
        for p in protein_list:
            self.fc3[p] = nn.Linear(256, 64)
        self.fc4 = nn.ModuleDict({})
        for p in protein_list:
            self.fc4[p] = nn.Linear(64, 1)
#
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        outputs = {}
        for p in protein_list:
            outputs[p] = self.fc4[p](F.relu(self.fc3[p](x)))
        return outputs

## Initialize network and set training hyperparameters

In [17]:
net = Net().to("cuda")
net

Net(
  (fc1): Linear(in_features=13553, out_features=1000, bias=True)
  (fc2): Linear(in_features=1000, out_features=256, bias=True)
  (fc3): ModuleDict(
    (ADT_CD339(Jagged1)_A0887): Linear(in_features=256, out_features=64, bias=True)
    (ADT_CD272(BTLA)_A0881): Linear(in_features=256, out_features=64, bias=True)
    (ADT_ESAM_A0596): Linear(in_features=256, out_features=64, bias=True)
    (ADT_CD63_A0559): Linear(in_features=256, out_features=64, bias=True)
    (ADT_CD200(OX2)_A0079): Linear(in_features=256, out_features=64, bias=True)
    (ADT_CD197(CCR7)_A0377): Linear(in_features=256, out_features=64, bias=True)
    (ADT_CD2_A0892): Linear(in_features=256, out_features=64, bias=True)
    (ADT_PIR-A-B_A0882): Linear(in_features=256, out_features=64, bias=True)
    (ADT_CD278(ICOS)_A0171): Linear(in_features=256, out_features=64, bias=True)
    (ADT_CD273(B7-DC_PD-L2)_A0914): Linear(in_features=256, out_features=64, bias=True)
    (ADT_CD124(IL-4Ra)_A0916): Linear(in_features=256

In [18]:
criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters(), lr=0.001, amsgrad=True, weight_decay=0.001)
max_epochs = 200
n_batches = 32 
patience = 30
best_score = None

Dy = len(protein_list)
estop_counter = pd.Series(np.zeros(Dy), index=protein_list)
early_stop = pd.Series([False] * Dy, index=protein_list)

train_loss = pd.DataFrame(np.zeros(shape=(len(protein_list), max_epochs)), index=protein_list)
test_loss = pd.DataFrame(np.zeros(shape=(len(protein_list), max_epochs)), index=protein_list)

## Start training

In [19]:
for epoch in range(max_epochs):
    if all(early_stop):
        break
    running_loss = pd.Series(np.zeros(Dy), index=protein_list)
    for i in range(int(y.shape[0] / n_batches)):
        local_X, local_y = X[i * n_batches:min((i + 1) * n_batches, X.shape[0] - 1), ], y[i * n_batches:min((i + 1) * n_batches, y.shape[0] - 1), ]
        optimizer.zero_grad()
        outputs_dict = net(local_X)
        loss = None
        for p in protein_list:
            loss_p = criterion(outputs_dict[p], local_y[:, protein_list.index(p):(protein_list.index(p) + 1)])
            if not torch.isnan(loss_p):
                running_loss[p] += loss_p.item()
                if loss is None:
                    loss = loss_p
                else:
                    loss = loss + loss_p
        loss.backward()
        optimizer.step()
        
    avg_loss = (running_loss / int(y.shape[0] / n_batches)).sum()
    train_loss.iloc[:, epoch] = (running_loss / int(y.shape[0] / n_batches))
    print(f"Epoch [{epoch+1}/{max_epochs}], Loss: {avg_loss:.4f}")
    
    test_outputs = net(X_test)   
    test_outputs = [test_outputs[p] for p in protein_list]
    test_outputs = torch.transpose(torch.stack(test_outputs), 0, 1).view(X_test.shape[0], -1)
    test_loss_i = pd.Series([criterion(test_outputs[:, pi], y_test[:, pi]).item() for pi in range(Dy)], index=protein_list)
    test_loss.iloc[:, epoch] = test_loss_i
    if best_score is None:
        best_score = test_loss_i
    else:
        for p in protein_list:
            if test_loss_i[p] > (best_score[p] - 0.001) and (not early_stop[p]):
                estop_counter[p] += 1
                if estop_counter[p] >= patience:
                    early_stop[p] = True
            else:
                best_score[p] = test_loss_i[p]
                estop_counter[p] = 0

Epoch [1/200], Loss: 62.4040
Epoch [2/200], Loss: 53.3566
Epoch [3/200], Loss: 52.2753
Epoch [4/200], Loss: 51.4350
Epoch [5/200], Loss: 50.9515
Epoch [6/200], Loss: 50.6380
Epoch [7/200], Loss: 50.4083
Epoch [8/200], Loss: 50.0918
Epoch [9/200], Loss: 49.8231
Epoch [10/200], Loss: 49.6329
Epoch [11/200], Loss: 49.4754
Epoch [12/200], Loss: 49.2395
Epoch [13/200], Loss: 49.2864
Epoch [14/200], Loss: 48.9536
Epoch [15/200], Loss: 48.8082
Epoch [16/200], Loss: 48.6559
Epoch [17/200], Loss: 48.4677
Epoch [18/200], Loss: 48.3256
Epoch [19/200], Loss: 48.1040
Epoch [20/200], Loss: 48.0008
Epoch [21/200], Loss: 47.8998
Epoch [22/200], Loss: 47.9279
Epoch [23/200], Loss: 48.0963
Epoch [24/200], Loss: 48.0137
Epoch [25/200], Loss: 47.7162
Epoch [26/200], Loss: 47.8139
Epoch [27/200], Loss: 47.6527
Epoch [28/200], Loss: 47.5385
Epoch [29/200], Loss: 47.6414
Epoch [30/200], Loss: 47.8437
Epoch [31/200], Loss: 47.6562
Epoch [32/200], Loss: 47.4075
Epoch [33/200], Loss: 47.3110
Epoch [34/200], Los

## Save the loss during training

In [20]:
train_loss = train_loss.iloc[:, :epoch]
test_loss = test_loss.iloc[:, :epoch]

torch.save(net.state_dict(), f"{base_dir}/model.pth")
train_loss.to_csv(f"{base_dir}/training_loss.csv")
test_loss.to_csv(f"{base_dir}/test_loss.csv")
train_loss, test_loss

(                                0         1         2         3         4   \
 ADT_CD339(Jagged1)_A0887  0.135074  0.126472  0.126466  0.124513  0.123386   
 ADT_CD272(BTLA)_A0881     0.237322  0.204731  0.203067  0.201280  0.199983   
 ADT_ESAM_A0596            0.276343  0.268064  0.267212  0.263390  0.262368   
 ADT_CD63_A0559            0.192539  0.177280  0.175262  0.173157  0.172003   
 ADT_CD200(OX2)_A0079      0.234429  0.207769  0.202624  0.200187  0.197933   
 ...                            ...       ...       ...       ...       ...   
 ADT_CD83_A0562            0.176228  0.155453  0.152500  0.149385  0.147426   
 ADT_CD26(DPP-4)_A0883     0.124923  0.094728  0.091342  0.088681  0.087231   
 ADT_ENPP1_A0891           0.168174  0.164748  0.161932  0.161262  0.160547   
 ADT_CD150(SLAM)_A0203     0.346643  0.295721  0.287815  0.279006  0.273130   
 ADT_CD155(PVR)_A1011      0.334776  0.300492  0.290405  0.284636  0.282101   
 
                                 5         6      

## Normalize query raw gene expression

In [21]:
X_pred_raw = x_batch2
X_pred = X_pred_raw.loc[:, gene_list]
X_pred = X_pred.apply(lambda x: np.log((x * 10000.0 / sum(x)) + 1.0), axis=1)
X_pred

index,0610007P14Rik,0610009B22Rik,0610009L18Rik,0610009O20Rik,0610010F05Rik,0610010K14Rik,0610011F06Rik,0610012G03Rik,0610030E20Rik,0610037L13Rik,...,mt-Co2,mt-Co3,mt-Cytb,mt-Nd1,mt-Nd2,mt-Nd3,mt-Nd4,mt-Nd4l,mt-Nd5,mt-Nd6
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGCGATCGA-1,0.242766,0.699239,3.661876,0.001771,0.188476,0.063529,0.010580,0.094648,0.290291,0.278284,...,0.244156,0.339453,0.900453,0.120123,0.055176,0.573618,0.369302,0.290291,0.245544,4.916514
AAACCCACACCTCTAC-1,0.471157,0.672287,3.141607,0.003851,0.203506,0.020997,0.011508,0.104318,0.045256,0.043411,...,0.273400,0.267513,1.337267,0.450473,0.081466,0.192428,0.401339,0.466329,0.212905,4.754729
AAACCCACACGTCGTG-1,0.435586,1.030374,3.222365,0.139373,0.197802,0.089357,0.011154,0.099562,0.122977,0.116343,...,0.187007,0.386005,1.062541,0.177660,0.087646,0.247182,0.331156,0.502244,0.301178,4.987180
AAACCCACATGTCTAG-1,0.447606,0.849128,3.394511,0.043327,0.195453,0.048613,0.011009,0.098323,0.060840,0.050369,...,0.206019,0.285367,1.161750,0.269993,0.059103,0.206019,0.207520,0.396810,0.255809,5.674700
AAACCCAGTCAAAGTA-1,0.458393,0.848226,3.563082,0.005488,0.194443,0.021775,0.010947,0.099452,0.097790,0.034261,...,0.195952,0.212404,1.139806,0.294941,0.051833,0.188383,0.203464,0.319230,0.275633,5.241976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,0.546502,1.011894,3.235216,0.010986,0.196586,0.036159,0.010986,0.101454,0.076186,0.034381,...,0.179809,0.208613,1.084175,0.312152,0.055502,0.159612,0.405969,0.399816,0.302675,4.807837
TTTGTTGGTGCTCTTC-1,0.494792,0.727922,3.154365,0.007660,0.204444,0.046941,0.011468,0.103978,0.105709,0.052428,...,0.241360,0.328097,1.045737,0.435645,0.091776,0.272581,0.305693,0.434400,0.337743,5.304375
TTTGTTGTCAGCCTTC-1,0.252563,0.658267,3.361503,0.001876,0.198618,0.063657,0.091460,0.101691,0.239347,0.384907,...,0.433544,0.355073,1.282748,0.219943,0.179969,0.321603,0.574125,0.410144,0.333782,4.612128


## Convert gene expression to Torch Tensor

In [22]:
X_pred = torch.tensor(X_pred.values)
X_pred = X_pred.type(torch.FloatTensor)
X_pred = X_pred.to("cuda")
X_pred

tensor([[0.2428, 0.6992, 3.6619,  ..., 0.2903, 0.2455, 4.9165],
        [0.4712, 0.6723, 3.1416,  ..., 0.4663, 0.2129, 4.7547],
        [0.4356, 1.0304, 3.2224,  ..., 0.5022, 0.3012, 4.9872],
        ...,
        [0.2526, 0.6583, 3.3615,  ..., 0.4101, 0.3338, 4.6121],
        [0.3134, 0.5319, 3.4879,  ..., 0.4642, 0.2925, 5.1728],
        [0.4297, 0.5032, 2.9203,  ..., 0.6497, 0.2013, 4.7696]],
       device='cuda:0')

## Perform prediction on the query

In [23]:
test_outputs_ = net(X_pred)

def fun(x):
    test_outputs_[x] = test_outputs_[x].detach().cpu().numpy().flatten()

ing = [fun(i) for i in test_outputs_]
test_outputs_ = pd.DataFrame.from_dict(test_outputs_, orient='index')
test_outputs_.columns = X_pred_raw.index
test_outputs_ = test_outputs_.T
test_outputs_.to_csv(f"{base_dir}/test_predicted_protein_expression (cTPnet).csv")
test_outputs_

,ADT_CD339(Jagged1)_A0887,ADT_CD272(BTLA)_A0881,ADT_ESAM_A0596,ADT_CD63_A0559,ADT_CD200(OX2)_A0079,ADT_CD197(CCR7)_A0377,ADT_CD2_A0892,ADT_PIR-A-B_A0882,ADT_CD278(ICOS)_A0171,ADT_CD273(B7-DC_PD-L2)_A0914,...,ADT_CD371(CLEC12A)_A0825,ADT_CD106_A0226,ADT_CD1d(CD1/1_Ly-38)_A0851,ADT_FceRIa_A0115,ADT_PLZF_A0878,ADT_CD83_A0562,ADT_CD26(DPP-4)_A0883,ADT_ENPP1_A0891,ADT_CD150(SLAM)_A0203,ADT_CD155(PVR)_A1011
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGCGATCGA-1,0.701519,-0.342424,-0.436930,0.174096,0.097218,-0.087761,2.530185,-0.323219,0.586917,-0.779938,...,-0.748734,-0.361084,0.048087,-0.608738,-0.478255,-1.406853,1.839212,0.294416,0.165791,-0.248224
AAACCCACACCTCTAC-1,0.758352,0.153912,-0.500211,0.044813,0.844480,-0.260867,1.999589,0.510575,-0.007724,-0.978873,...,-0.053089,-0.301036,2.404577,-0.841118,-0.578377,-1.697636,1.358449,0.108661,0.955528,-0.232259
AAACCCACACGTCGTG-1,0.771610,0.374123,-0.395313,0.205485,0.791792,-0.078035,2.479775,0.659072,0.107209,-0.729224,...,-0.065132,-0.250608,1.726287,-0.571043,-0.673993,-1.602600,1.503769,0.283097,0.816887,-0.405219
AAACCCACATGTCTAG-1,0.729030,0.451394,-0.478414,0.151023,1.024541,-0.186371,2.436165,0.679903,0.099520,-0.760503,...,-0.284461,-0.397807,1.612529,-0.776030,-0.659756,-1.587443,1.415689,0.228878,1.168741,-0.418235
AAACCCAGTCAAAGTA-1,0.706716,0.426363,-0.498750,0.102534,1.353212,-0.278737,2.365363,0.326206,0.074335,-0.665942,...,-0.416423,-0.258481,1.091699,-0.821857,-0.675624,-1.586425,1.317187,0.053390,1.368811,-0.403096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,0.742271,0.572464,-0.468024,0.172310,1.387028,-0.150892,2.357247,0.194971,0.130066,-0.759001,...,-0.476170,-0.318289,0.853597,-0.714117,-0.563140,-1.502131,1.313081,0.247673,1.388466,-0.441969
TTTGTTGGTGCTCTTC-1,0.729698,0.576195,-0.471309,0.194047,1.369689,-0.232933,2.422460,0.161858,0.144458,-0.779889,...,-0.449192,-0.248720,0.747555,-0.794246,-0.554636,-1.576740,1.292380,0.270026,1.368184,-0.419234
TTTGTTGTCAGCCTTC-1,0.764950,-0.165404,-0.451516,0.209819,0.211336,-0.086858,2.494130,-0.293906,0.547543,-0.705775,...,-0.860249,-0.357543,0.220537,-0.543087,-0.400196,-1.428550,1.405389,0.235628,0.543697,0.648083
